# Tutorial 16 — Bayesian Interpretation of HLLSet Algebra

This capstone tutorial introduces a **second interpretation** of the HLLSet
framework: the **Bayesian probabilistic view**. It complements — and sometimes
*competes with* — the **evolutionary view** developed in Tutorials 13–15.

## The Central Thesis

Every operation in the HLLSet algebra admits **two readings**:

| Aspect | Evolution Interpretation | Bayesian Interpretation |
|--------|------------------------|------------------------|
| State $R(t)$ | Physical system state | Belief state |
| Transition $R(t) \to R(t{+}1)$ | System changed | Beliefs updated |
| $|A|$ | How many elements A has | How *certain* we are of A |
| $\Phi(t) = |N| - |D|$ | Growth / Decay | Evidence balance |
| Conservation (popcount) | Physical invariant | Normalization constraint |
| Universe $U$ | Not needed — state is absolute | Essential — defines context |
| Branch (EvolutionGraph) | Parallel reality | Competing hypothesis |
| Merge | Combine histories | Model averaging |

These two interpretations are **complementary** — like wave and particle
descriptions of the same phenomenon. But they can also **compete**:
the evolution view may say "stable" while the Bayesian view says
"dramatic shift." Exploring when and why they diverge is the
deepest insight of this tutorial.

## Prerequisites

| Tutorial | Concept Used Here |
|----------|------------------|
| [01_hllset.ipynb](01_hllset.ipynb) | `HLLSet`, `union`, `intersect`, `diff`, `cardinality` |
| [03_hll_lattice.ipynb](03_hll_lattice.ipynb) | `HLLLattice`, `cumulative`, `delta` |
| [08_debruijn.ipynb](08_debruijn.ipynb) | `DeBruijnGraph`, multiplicities, Eulerian paths |
| [13_observe.ipynb](13_observe.ipynb) | `NoetherEvolution`, flux $\Phi(t)$ |
| [14_act.ipynb](14_act.ipynb) | `EvolutionGraph`, branching, merge |

In [1]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, math
sys.path.insert(0, '..')

from core.hllset import HLLSet
from core.hll_lattice import HLLLattice
from core.noether import NoetherEvolution, compute_flux, apply_transition
from core.bayesian import (
    prior, conditional, joint, bayes_theorem,
    surprise, entropy_of_partition, kl_divergence,
    bayesian_surprise_temporal,
    temporal_posterior, temporal_trajectory,
    interpretation_divergence,
)

P_BITS = 10
print('Imports OK')

Imports OK


---
## Part I — Bayesian Foundations on HLLSets
*(Complements Tutorial 01: core set operations)*

### 1. Prior Probability $P(A)$

The most basic Bayesian quantity. Given a universe $U$, the prior
probability of entity $A$ is:

$$P(A) = \frac{|A|}{|U|}$$

**Key insight from the SGS document**: there is no "universal" universe.
The *same* HLLSet $A$ has different probabilities under different universes.

In [2]:
# Three entities with overlapping tokens (same as the Bayesian document's Julia example)
A = HLLSet.from_batch([f"string{i}" for i in range(11)])       # 0..10
B = HLLSet.from_batch([f"string{i}" for i in range(3, 12)])    # 3..11
C = HLLSet.from_batch([f"string{i}" for i in range(5, 12)])    # 5..11

# The universe is the union of all entities
U = HLLSet.merge([A, B, C])

print(f"Cardinalities:  |A|≈{A.cardinality():.0f}  |B|≈{B.cardinality():.0f}  "
      f"|C|≈{C.cardinality():.0f}  |U|≈{U.cardinality():.0f}")
print()

# Priors
for name, entity in [('A', A), ('B', B), ('C', C)]:
    r = prior(entity, U)
    print(f"P({name}) = {r.value:.4f}   ({r.description})")

Cardinalities:  |A|≈13  |B|≈10  |C|≈9  |U|≈14

P(A) = 0.9286   (P(A) = |A|/|U| = 13.0/14.0)
P(B) = 0.7143   (P(A) = |A|/|U| = 10.0/14.0)
P(C) = 0.6429   (P(A) = |A|/|U| = 9.0/14.0)


### 2. Conditional Probability $P(A|B)$

$$P(A|B) = \frac{|A \cap B|}{|B|}$$

"Given that we observe $B$, what is the probability that $A$ is also present?"

In [3]:
# Conditional probabilities (compare with Julia output in the document)
pairs = [('A','B',A,B), ('B','A',B,A), ('A','C',A,C), 
         ('C','A',C,A), ('B','C',B,C), ('C','B',C,B)]

for n1, n2, e1, e2 in pairs:
    r = conditional(e1, e2)
    print(f"P({n1}|{n2}) = {r.value:.4f}")

P(A|B) = 0.9000
P(B|A) = 0.6923
P(A|C) = 0.8889
P(C|A) = 0.6154
P(B|C) = 1.0000
P(C|B) = 0.9000


### 3. Bayes' Theorem Verification

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

With exact sets, both sides are identical. With HLLSets, the probabilistic
cardinality introduces a small **consistency error** — this *is* the
epistemic uncertainty inherent in the HLL approximation.

In [4]:
# Verify Bayes' theorem for each pair
for n1, n2, e1, e2 in [('A','B',A,B), ('A','C',A,C), ('B','C',B,C)]:
    bt = bayes_theorem(e1, e2, U)
    status = '✓' if bt.is_consistent() else '✗'
    print(f"Bayes({n1},{n2}): P({n1}|{n2})={bt.p_a_given_b:.4f}  "
          f"RHS={bt.bayes_rhs:.4f}  error={bt.consistency_error:.4f}  {status}")

Bayes(A,B): P(A|B)=0.9000  RHS=0.9000  error=0.0000  ✓
Bayes(A,C): P(A|C)=0.8889  RHS=0.8889  error=0.0000  ✓
Bayes(B,C): P(B|C)=1.0000  RHS=1.0000  error=0.0000  ✓


### 4. Shannon Surprise and Entropy

**Surprise**: $S(A) = -\log_2 P(A)$  
Low probability → high surprise. A rare entity carries more information.

**Entropy**: $H = -\sum_i P(A_i) \log_2 P(A_i)$  
Maximum entropy = maximum uncertainty = uniform distribution.

In [5]:
# Surprise of each entity
for name, entity in [('A', A), ('B', B), ('C', C)]:
    s = surprise(entity, U)
    p = prior(entity, U).value
    print(f"S({name}) = {s:.4f} bits   (P={p:.4f})")

# Entropy of the 3-entity partition
H = entropy_of_partition([A, B, C], U)
H_max = math.log2(3)  # Maximum for 3 entities
print(f"\nPartition entropy: H = {H:.4f} bits")
print(f"Maximum possible:  H_max = {H_max:.4f} bits")
print(f"Normalized:        H/H_max = {H/H_max:.4f}")

S(A) = 0.1069 bits   (P=0.9286)
S(B) = 0.4854 bits   (P=0.7143)
S(C) = 0.6374 bits   (P=0.6429)

Partition entropy: H = 0.8558 bits
Maximum possible:  H_max = 1.5850 bits
Normalized:        H/H_max = 0.5399


---
## Part II — Temporal Bayesian Updating
*(Complements Tutorial 03: HLLLattice temporal queries)*

### 5. Local Universes and Shifting Priors

The SGS document emphasizes that the universe is not fixed:

> *"There is no 'universal' universe. What was deemed acceptable
> a few years ago may no longer hold today."*

Using the lattice, we track how $P(A)$ changes as the universe evolves.
This is the **temporal posterior**: $P_t(A) = |A \cap U(t)| / |U(t)|$.

In [6]:
# Build a lattice with evolving content
lattice = HLLLattice(p_bits=P_BITS)

# t=1: small universe, entity A dominates
obs1 = HLLSet.from_batch([f"string{i}" for i in range(5)])          # 5 tokens
lattice.append([obs1], timestamp=1.0)

# t=2: universe grows moderately
obs2 = HLLSet.from_batch([f"string{i}" for i in range(3, 10)])      # overlap + new
lattice.append([obs2], timestamp=2.0)

# t=3: universe grows dramatically with new domain
obs3 = HLLSet.from_batch([f"new_domain_{i}" for i in range(50)])    # 50 new tokens
lattice.append([obs3], timestamp=3.0)

# t=4: yet more new content
obs4 = HLLSet.from_batch([f"another_domain_{i}" for i in range(100)])
lattice.append([obs4], timestamp=4.0)

# Track entity A across time
entity_A = HLLSet.from_batch([f"string{i}" for i in range(5)])

print("Temporal posterior P_t(A) — same entity, shifting universe:")
print("-" * 60)
trajectory = temporal_trajectory(lattice, entity_A, [1.0, 2.0, 3.0, 4.0])
for rec in trajectory:
    print(f"  t={rec.timestamp:.0f}: P(A)={rec.p_a:.4f}  |U|≈{rec.universe_card:.0f}  "
          f"surprise={rec.surprise:.2f} bits")

Temporal posterior P_t(A) — same entity, shifting universe:
------------------------------------------------------------
  t=1: P(A)=1.0000  |U|≈6  surprise=0.00 bits
  t=2: P(A)=0.5000  |U|≈12  surprise=1.00 bits
  t=3: P(A)=0.0923  |U|≈65  surprise=3.44 bits
  t=4: P(A)=0.0351  |U|≈171  surprise=4.83 bits


**Observation**: Entity A's *absolute* content never changed — it is always the
same 5 tokens. But its *probability* dropped as the universe grew.

The evolution interpretation would say "nothing happened to A."
The Bayesian interpretation says "A became increasingly irrelevant."

This is the first glimpse of **interpretive divergence**.

### 6. KL Divergence Between Universes

How much did our *entire belief system* change between time steps?

$$D_{KL}(P_t \| P_{t+1}) = \sum_i P_t(A_i) \log_2 \frac{P_t(A_i)}{P_{t+1}(A_i)}$$

This measures the "cost" of updating from the old universe to the new one.
The evolution interpretation has no direct analogue — it tracks flux,
not belief revision.

In [7]:
# Define a few entities to track
entities = [
    HLLSet.from_batch([f"string{i}" for i in range(5)]),
    HLLSet.from_batch([f"string{i}" for i in range(3, 10)]),
    HLLSet.from_batch([f"new_domain_{i}" for i in range(20)]),
]

# KL divergence between consecutive universes
timestamps = [1.0, 2.0, 3.0, 4.0]
print("KL Divergence between consecutive universes:")
print("-" * 50)
for i in range(len(timestamps) - 1):
    u_before = lattice.cumulative(t=timestamps[i])
    u_after = lattice.cumulative(t=timestamps[i + 1])
    kl = kl_divergence(entities, u_before, u_after)
    print(f"  t={timestamps[i]:.0f}→{timestamps[i+1]:.0f}: "
          f"D_KL = {kl:.4f} bits  "
          f"(|U| {u_before.cardinality():.0f}→{u_after.cardinality():.0f})")

KL Divergence between consecutive universes:
--------------------------------------------------
  t=1→2: D_KL = 6.3333 bits  (|U| 6→12)
  t=2→3: D_KL = 7.7185 bits  (|U| 12→65)
  t=3→4: D_KL = 0.8158 bits  (|U| 65→171)


---
## Part III — Bayesian Path Selection for De Bruijn Graphs
*(Complements Tutorial 08: sequence reconstruction)*

### 7. Edges as Evidence

In Tutorial 08, we reconstruct sequences from De Bruijn graphs using
Eulerian paths. The evolution interpretation treats all valid paths
as structurally equivalent.

The Bayesian interpretation says: **not all paths are equally likely.**
Edge multiplicities encode frequency → frequency encodes probability.

$$P(\text{path}) = \prod_i P(\text{edge}_i) = \prod_i \frac{m_i}{\sum m_j}$$

In [8]:
from core.debruijn import DeBruijnGraph
from core.bayesian import edge_probability, path_log_likelihood

# Build a graph where some transitions are more common
graph = DeBruijnGraph(k=3)

# Frequent pattern: A→B→C (appears 5 times)
for _ in range(5):
    graph.add_kmer(('<S>', 'A', 'B'))
    graph.add_kmer(('A', 'B', 'C'))
    graph.add_kmer(('B', 'C', '</S>'))

# Rare pattern: A→B→D (appears once)
graph.add_kmer(('<S>', 'A', 'B'))
graph.add_kmer(('A', 'B', 'D'))
graph.add_kmer(('B', 'D', '</S>'))

total = graph.total_edge_count
print(f"Graph: {graph.num_edges} unique edges, {total} total (with multiplicity)")
print()

# Compare edge probabilities
for kmer in [('A','B','C'), ('A','B','D')]:
    e = graph.get_edge(kmer)
    p = edge_probability(e.multiplicity, total)
    print(f"  Edge {kmer}: mult={e.multiplicity}, P={p:.4f}")

print(f"\nBayesian verdict: transition A→B→C is "
      f"{graph.get_edge(('A','B','C')).multiplicity}x more likely than A→B→D")
print("Evolution verdict: both are structurally valid edges (no preference).")

Graph: 5 unique edges, 18 total (with multiplicity)

  Edge ('A', 'B', 'C'): mult=5, P=0.2778
  Edge ('A', 'B', 'D'): mult=1, P=0.0556

Bayesian verdict: transition A→B→C is 5x more likely than A→B→D
Evolution verdict: both are structurally valid edges (no preference).


---
## Part IV — Complementary Nature: Two Lenses, One Reality
*(Connects Tutorials 01, 03, 08 with Tutorials 13–15)*

### 8. The State Transition Has Two Meanings

The Noether state transition:

$$R(t{+}1) = \bigl[R(t) \setminus D(t)\bigr] \cup N(t)$$

**Evolution reading** (Tutorial 13):
- $R(t)$ is the system state (a physical fact)
- $D(t)$ are removed elements (system lost information)
- $N(t)$ are added elements (system gained information)
- Conservation: if $|N| = |D|$, popcount is invariant

**Bayesian reading** (this tutorial):
- $R(t)$ is the belief state (what we currently think is true)
- $D(t)$ is evidence *against* (disconfirmation)
- $N(t)$ is evidence *for* (confirmation)
- The universe $U(t)$ provides the normalization context
- $P_t(A) = |A \cap R(t)| / |U(t)|$ is the posterior

Both readings use **exactly the same operations** — `diff`, `union`,
`intersect`, `cardinality`. The difference is purely in *interpretation*.

In [9]:
# Demonstrate: same transition, two readings
evo = NoetherEvolution(p_bits=P_BITS)

# Step 1: add some initial content
initial = HLLSet.from_batch([f"token_{i}" for i in range(20)])
diag = evo.step(additions=initial)

# Step 2: balanced update (remove 5, add 5 different)
deletions = HLLSet.from_batch([f"token_{i}" for i in range(5)])
additions = HLLSet.from_batch([f"new_token_{i}" for i in range(5)])

state_before = evo.state
universe_before = state_before  # Universe = current state

diag = evo.step(additions=additions, deletions=deletions)

state_after = evo.state
universe_after = state_after

print("═" * 60)
print("EVOLUTION VIEW:")
print(f"  Flux Φ = {diag.flux:+.1f}")
print(f"  Phase: {diag.phase.value}")
print(f"  Popcount: {diag.popcount}")
print(f"  Verdict: {diag.phase.value} — conservation {'holds' if diag.is_balanced() else 'broken'}")

print()
print("BAYESIAN VIEW:")
# Track an entity that was partially affected
entity = HLLSet.from_batch([f"token_{i}" for i in range(10)])  # half old tokens
p_before = prior(entity, universe_before).value
p_after = prior(entity, universe_after).value
print(f"  P_before(entity) = {p_before:.4f}")
print(f"  P_after(entity)  = {p_after:.4f}")
print(f"  ΔP = {p_after - p_before:+.4f}")
print(f"  Verdict: {'strengthened' if p_after > p_before else 'weakened' if p_after < p_before else 'stable'}")
print("═" * 60)

════════════════════════════════════════════════════════════
EVOLUTION VIEW:
  Flux Φ = +0.0
  Phase: balanced
  Popcount: 20
  Verdict: balanced — conservation broken

BAYESIAN VIEW:
  P_before(entity) = 0.5000
  P_after(entity)  = 0.5000
  ΔP = +0.0000
  Verdict: stable
════════════════════════════════════════════════════════════


### 9. What Each Interpretation Reveals (and Hides)

| Question | Evolution Answers | Bayesian Answers |
|----------|------------------|------------------|
| How much did the system change? | $\Phi(t)$ flux, popcount Δ | $D_{KL}$ divergence |
| Is the system healthy? | Conservation check | Entropy trend |
| What is entity A's status? | $|A|$ cardinality | $P(A|U)$ probability |
| Are entities A and B related? | $|A \cap B|$ overlap | $P(A|B)$ conditional |
| Which path is most likely? | All Eulerian paths equal | $\prod P(\text{edge}_i)$ |
| What should we do next? | Steer to minimize $|\Phi|$ | Maximize expected information gain |

**Neither interpretation is "more correct"** — they illuminate different
aspects of the same mathematical structure. A complete understanding
requires both.

---
## Part V — Competition: When Interpretations Diverge

### 10. The Dilution Effect

**Evolution says: growth** ($\Phi > 0$, system expanding)  
**Bayesian says: weakening** ($P(A)$ decreasing)

This happens when the universe grows *faster* than the entity.
Like a company whose revenue grows but market share shrinks.

In [10]:
# DILUTION: Evolution sees growth, Bayesian sees weakening
evo_dilution = NoetherEvolution(p_bits=P_BITS)

# Start with entity A as a large fraction of the state
entity_A = HLLSet.from_batch([f"entity_a_{i}" for i in range(20)])
evo_dilution.step(additions=entity_A)
state_before = evo_dilution.state

# Add MANY new tokens that are NOT in entity A
flood = HLLSet.from_batch([f"flood_{i}" for i in range(200)])
diag = evo_dilution.step(additions=flood)
state_after = evo_dilution.state

comp = interpretation_divergence(
    entity=entity_A,
    state_before=state_before, state_after=state_after,
    universe_before=state_before, universe_after=state_after,
    additions=flood, deletions=None,
)

print("DILUTION SCENARIO")
print("=" * 60)
print(f"Evolution: {comp.evolution_verdict} (Φ={comp.flux:+.1f})")
print(f"Bayesian:  {comp.bayesian_verdict} (P: {comp.prior_probability:.4f} → {comp.posterior_probability:.4f})")
print(f"Agree? {comp.interpretations_agree}")
print(f"\n{comp.divergence_note}")

DILUTION SCENARIO
Evolution: growth (Φ=+194.0)
Bayesian:  weakened (P: 1.0000 → 0.0977)
Agree? False

DILUTION: Evolution sees growth (Φ>0), but Bayesian sees weakening. The universe grew faster than the entity — absolute growth, relative decline. Like a company whose revenue grows but market share shrinks.


### 11. The Concentration Effect

**Evolution says: decay** ($\Phi < 0$, system contracting)  
**Bayesian says: strengthening** ($P(A)$ increasing)

This happens when the universe shrinks *faster* than the entity.
Like a company that loses revenue but gains market share
in a collapsing market.

In [11]:
# CONCENTRATION: Evolution sees decay, Bayesian sees strengthening
evo_conc = NoetherEvolution(p_bits=P_BITS)

# Start with a large state: entity A + lots of other stuff
entity_A = HLLSet.from_batch([f"entity_a_{i}" for i in range(20)])
other = HLLSet.from_batch([f"other_{i}" for i in range(200)])
combined_init = entity_A.union(other)
evo_conc.step(additions=combined_init)
state_before = evo_conc.state

# Remove most of the "other" stuff, leaving entity A relatively intact
mass_delete = HLLSet.from_batch([f"other_{i}" for i in range(180)])
diag = evo_conc.step(deletions=mass_delete)
state_after = evo_conc.state

comp = interpretation_divergence(
    entity=entity_A,
    state_before=state_before, state_after=state_after,
    universe_before=state_before, universe_after=state_after,
    additions=None, deletions=mass_delete,
)

print("CONCENTRATION SCENARIO")
print("=" * 60)
print(f"Evolution: {comp.evolution_verdict} (Φ={comp.flux:+.1f})")
print(f"Bayesian:  {comp.bayesian_verdict} (P: {comp.prior_probability:.4f} → {comp.posterior_probability:.4f})")
print(f"Agree? {comp.interpretations_agree}")
print(f"\n{comp.divergence_note}")

CONCENTRATION SCENARIO
Evolution: decay (Φ=-170.0)
Bayesian:  strengthened (P: 0.0995 → 0.5122)
Agree? False

CONCENTRATION: Evolution sees decay (Φ<0), but Bayesian sees strengthening. The universe shrank faster than the entity — absolute decline, relative growth. Like a company that loses revenue but gains market share in a shrinking market.


### 12. The Hidden Shift

**Evolution says: balanced** ($\Phi \approx 0$, popcount conserved)  
**Bayesian says: dramatic change** (probabilities shifted)

This is the most *philosophically interesting* case. The Noether
conservation law is satisfied — the physicist sees a stable system.
But the Bayesian sees the *composition* of the universe change:
old certainties eroded, new beliefs formed.

Like a balanced budget where total spending is constant but the
priorities shifted from defense to education.

In [12]:
# HIDDEN SHIFT: Evolution balanced, Bayesian sees change
evo_shift = NoetherEvolution(p_bits=P_BITS)

# Start with 50 tokens: 25 from domain_A, 25 from domain_B
domain_a_tokens = [f"domain_a_{i}" for i in range(25)]
domain_b_tokens = [f"domain_b_{i}" for i in range(25)]
initial = HLLSet.from_batch(domain_a_tokens + domain_b_tokens)
evo_shift.step(additions=initial)
state_before = evo_shift.state

# Balanced update: remove 15 from domain_A, add 15 from domain_C
# Total stays ~50, but composition shifts dramatically
deletions = HLLSet.from_batch([f"domain_a_{i}" for i in range(15)])
additions = HLLSet.from_batch([f"domain_c_{i}" for i in range(15)])
diag = evo_shift.step(additions=additions, deletions=deletions)
state_after = evo_shift.state

# Track domain_A as our entity of interest
entity_domain_a = HLLSet.from_batch(domain_a_tokens)

comp = interpretation_divergence(
    entity=entity_domain_a,
    state_before=state_before, state_after=state_after,
    universe_before=state_before, universe_after=state_after,
    additions=additions, deletions=deletions,
)

print("HIDDEN SHIFT SCENARIO")
print("=" * 60)
print(f"Evolution: {comp.evolution_verdict} (Φ={comp.flux:+.1f}, popcount Δ={comp.popcount_delta})")
print(f"Bayesian:  {comp.bayesian_verdict} (P: {comp.prior_probability:.4f} → {comp.posterior_probability:.4f})")
print(f"Agree? {comp.interpretations_agree}")
print(f"\n{comp.divergence_note}")

HIDDEN SHIFT SCENARIO
Evolution: balanced (Φ=+0.0, popcount Δ=0)
Bayesian:  stable (P: 0.5283 → 0.5283)
Agree? True

Both interpretations agree on the direction of change.


---
## Part VI — Speculative Framework: Duality or Competition?

### 13. A Taxonomy of Agreement and Disagreement

We can map the complete space of evolution × Bayesian verdicts:

```
                    Bayesian View
                    Strengthened    Stable    Weakened
              ┌──────────────┬──────────┬──────────────┐
    Growth    │  AGREEMENT   │  mild    │  DILUTION    │
  Evolution   │  (common)    │ dilution │  (diverge)   │
    View      ├──────────────┼──────────┼──────────────┤
    Balanced  │  HIDDEN      │AGREEMENT │  HIDDEN      │
              │  enrichment  │ (common) │  SHIFT       │
              ├──────────────┼──────────┼──────────────┤
    Decay     │CONCENTRATION │  mild    │  AGREEMENT   │
              │  (diverge)   │ conc.    │  (common)    │
              └──────────────┴──────────┴──────────────┘
```

The **diagonal** represents agreement. The **off-diagonal** cases
are where the two interpretations *compete* — and where the most
interesting insights hide.

### Speculation: Wave-Particle Duality of Information

This resembles **wave-particle duality** in quantum mechanics:

- The "wave" (Bayesian) description captures *interference* — how
  probabilities redistribute, how beliefs shift, how the composition
  of the universe matters even when its size is constant.

- The "particle" (Evolution) description captures *conservation* — how
  total information (popcount) is preserved, how flux measures
  absolute gain/loss, how the system state is unambiguous.

Neither alone is complete. The *complete* description of an HLLSet
system requires both — and their interplay reveals the deepest
structure of probabilistic set evolution.

### 14. A Multi-Step Evolution: Both Lenses at Once

Let us trace a 10-step evolution and compare the two interpretations
at each step, watching for agreement and divergence.

In [13]:
import random
random.seed(42)

# Initialize system
evo_dual = NoetherEvolution(p_bits=P_BITS)
initial_tokens = [f"base_{i}" for i in range(50)]
evo_dual.step(additions=HLLSet.from_batch(initial_tokens))

# Entity of interest: first 20 base tokens
entity = HLLSet.from_batch([f"base_{i}" for i in range(20)])

# Simulate 10 steps with varying patterns
scenarios = [
    ("balanced",   5,  5),   # Remove 5, add 5 new
    ("growth",     0,  20),  # Add 20, remove 0
    ("growth",     0,  30),  # Add 30 more
    ("balanced",  10, 10),   # Balanced swap
    ("decay",     25,  0),   # Remove 25
    ("growth",     0,  15),  # Add 15
    ("balanced",   8,  8),   # Balanced
    ("decay",     20,  5),   # Remove 20, add 5
    ("growth",     0,  40),  # Big growth
    ("balanced",  10, 10),   # Balanced
]

print(f"{'Step':>4} {'Scenario':>10} {'Evo Verdict':>12} {'Φ':>6} "
      f"{'Bayes Verdict':>14} {'P(A)':>8} {'ΔP':>8} {'Agree':>6}")
print("-" * 82)

step_counter = 0
for scenario_name, n_del, n_add in scenarios:
    step_counter += 1
    state_before = evo_dual.state
    
    # Build deletion set from random existing-style tokens
    del_tokens = [f"token_{random.randint(0, 200)}" for _ in range(n_del)] if n_del > 0 else []
    add_tokens = [f"token_{random.randint(200, 500)}" for _ in range(n_add)] if n_add > 0 else []
    
    dels = HLLSet.from_batch(del_tokens) if del_tokens else None
    adds = HLLSet.from_batch(add_tokens) if add_tokens else None
    
    diag = evo_dual.step(additions=adds, deletions=dels)
    state_after = evo_dual.state
    
    comp = interpretation_divergence(
        entity=entity,
        state_before=state_before, state_after=state_after,
        universe_before=state_before, universe_after=state_after,
        additions=adds, deletions=dels,
    )
    
    agree_sym = '✓' if comp.interpretations_agree else '✗'
    print(f"{step_counter:>4} {scenario_name:>10} {comp.evolution_verdict:>12} "
          f"{comp.flux:>+6.1f} {comp.bayesian_verdict:>14} "
          f"{comp.posterior_probability:>8.4f} {comp.probability_delta:>+8.4f} "
          f"{agree_sym:>6}")

Step   Scenario  Evo Verdict      Φ  Bayes Verdict     P(A)       ΔP  Agree
----------------------------------------------------------------------------------
   1   balanced        decay   -2.0       weakened   0.3448  -0.0325      ✓
   2     growth       growth  +19.0       weakened   0.2597  -0.0851      ✗
   3     growth       growth  +23.0       weakened   0.2000  -0.0597      ✗
   4   balanced     balanced   +1.0         stable   0.1887  -0.0113      ✓
   5      decay     balanced   +0.0         stable   0.1887  +0.0000      ✓
   6     growth       growth  +12.0         stable   0.1695  -0.0192      ✗
   7   balanced     balanced   -1.0         stable   0.1600  -0.0095      ✓
   8      decay        decay  -15.0         stable   0.1562  -0.0038      ✗
   9     growth       growth  +34.0       weakened   0.1235  -0.0328      ✗
  10   balanced       growth   +2.0         stable   0.1190  -0.0044      ✗


### 15. Synthesis: When to Use Which Interpretation

| Use Case | Preferred Interpretation | Why |
|----------|------------------------|-----|
| System health monitoring | **Evolution** | Conservation laws give clear pass/fail |
| Entity relevance tracking | **Bayesian** | Probability captures relative importance |
| Anomaly detection | **Both** | Evolution detects flux anomalies; Bayesian detects composition anomalies |
| Resource allocation | **Bayesian** | Probabilities map to priorities |
| History replay | **Evolution** | Git-like commits are structurally faithful |
| Hypothesis comparison | **Bayesian** | KL divergence compares belief states |
| Sequence reconstruction | **Both** | Eulerian validity (evo) + path likelihood (Bayes) |
| Regulatory compliance | **Evolution** | Conservation proofs are auditable |

### The Deeper Speculation

The competition between evolution and Bayesian interpretations may
reflect something fundamental about **information systems**:

1. **Absolute measures** (cardinality, popcount, flux) answer
   "what is?" — they describe the *state of the world*.

2. **Relative measures** (probability, surprise, KL divergence) answer
   "what does it mean?" — they describe the *state of knowledge*.

A Self-Generative System, by definition, transforms the world
through the act of observing it (each Commit changes the head).
This means **the act of measurement changes what is being measured** —
a direct parallel to the quantum measurement problem.

The evolution interpretation is the "Schrödinger picture" — the
state evolves, the operators are fixed.  
The Bayesian interpretation is the "Heisenberg picture" — the
observables (probabilities) evolve, the state is interpreted
relative to a shifting frame (the universe).

Both are exact. Both are incomplete. Together, they are sufficient.

---
## Summary

### What You Learned

1. **Bayesian primitives** — `prior`, `conditional`, `joint`, `bayes_theorem` on HLLSets
2. **Information measures** — surprise, entropy, KL divergence
3. **Temporal Bayesian updating** — probability trajectories over the lattice
4. **De Bruijn path selection** — edge probabilities from multiplicities
5. **Complementary interpretations** — same operations, different meanings
6. **Interpretation competition** — Dilution, Concentration, Hidden Shift
7. **Wave-particle duality of information** — Bayesian = wave, Evolution = particle

### API Reference

| Function | Purpose |
| ---------- | -------- |
| `prior(A, U)` | $P(A) = \|A\|/\|U\|$ |
| `conditional(A, B)` | $P(A\|B) = \|A \cap B\|/\|B\|$ |
| `joint(A, B, U)` | $P(A \cap B) = \|A \cap B\|/\|U\|$ |
| `bayes_theorem(A, B, U)` | Verify $P(A\|B) = P(B\|A) \cdot P(A)/P(B)$ |
| `surprise(A, U)` | $-\log_2 P(A)$ |
| `entropy_of_partition(entities, U)` | $H = -\sum P_i \log_2 P_i$ |
| `kl_divergence(entities, U_p, U_q)` | $D_{KL}(P \| Q)$ |
| `temporal_posterior(lattice, A, t)` | $P_t(A)$ using cumulative universe |
| `temporal_trajectory(lattice, A, ts)` | Time-series of $P_t(A)$ |
| `interpretation_divergence(...)` | Compare evolution vs Bayesian verdicts |
| `edge_probability(mult, total)` | De Bruijn edge prior |
| `path_log_likelihood(edges, graph)` | Path score for Bayesian reconstruction |

### The Grand Unification

$$\boxed{R(t{+}1) = \bigl[R(t) \setminus D(t)\bigr] \cup N(t)}$$

- **Tiers 1–3** define what $R$, $D$, $N$ *are*
- **Tier 4 (Evolution)** monitors *how much* changed → $\Phi$, popcount
- **Tier 5 (Bayesian)** interprets *what it means* → $P$, $H$, $D_{KL}$

The complete HLLSet Algebra framework is now a **dual-interpretation system**
where conservation laws (Noether) and probabilistic inference (Bayes)
provide complementary — and sometimes competing — views of the same
self-generative reality.